**Libraries**

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL

**Load Dataset**

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df.reset_index(drop=True, inplace=True)

# Quick look
print(df.head())
print(f"Shape: {df.shape}")
print(df.info())
print("Missing values:\n", df.isna().sum())
print((df.isna().sum() / len(df)) * 100)


**convert object columns**

In [ ]:
def clean_currency(x):
    """Remove commas and convert to float"""
    if isinstance(x, str):
        return float(x.replace(',', ''))
    return x

def clean_percent(x):
    """Convert percentage string to float"""
    if isinstance(x, str):
        return float(x.replace('%', '')) / 100
    return x

def clean_volume(x):
    """Convert volume strings like 1K, 1M, 1B to numeric"""
    if isinstance(x, str):
        x = x.upper()
        if 'K' in x:
            return float(x.replace('K',''))*1e3
        elif 'M' in x:
            return float(x.replace('M',''))*1e6
        elif 'B' in x:
            return float(x.replace('B',''))*1e9
        else:
            return float(x.replace(',',''))
    return x


gold_cols = ['Price_Gold','High_Gold','Low_Gold','Open_Gold']

# Clean gold price columns
for col in gold_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_currency)

# Clean percentage and volume
if 'Change%_Gold' in df.columns:
    df['Change%_Gold'] = df['Change%_Gold'].apply(clean_percent)
if 'Volume_Gold' in df.columns:
    df['Volume_Gold'] = df['Volume_Gold'].apply(clean_volume)

# Convert all gold columns to numeric (just in case)
for col in gold_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df.dtypes


NameError: name 'df' is not defined

 Basic Stats & Missing Values

In [ ]:
print(df.describe().T)
print(f"Number of duplicate rows: {df.duplicated().sum()}")

# Missing values heatmap
plt.figure(figsize=(12,6))
sns.heatmap(df.isna(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()

# First date with volume data
first_valid_index = df['Volume_Gold'].first_valid_index()
start_date_volume = df.loc[first_valid_index, 'Date']
print(f"The first date with available volume data is: {start_date_volume}")

# Missing percentage
missing_count = df['Volume_Gold'].isna().sum()
total_count = len(df)
print(f"Number of missing values: {missing_count} out of {total_count}")
print(f"Percentage of missing data: {(missing_count/total_count)*100:.2f}%")


**Outlier Detection**

In [ ]:
outlier_stats = {}
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask = (df[col] < lower) | (df[col] > upper)
    count = mask.sum()
    percent = (count / len(df)) * 100
    outlier_stats[col] = {"count": count, "percent": percent}

# Display
for col, stats in outlier_stats.items():
    print(f"- {col}: {stats['count']} outliers ({stats['percent']:.2f}%)")

# Boxplot
plt.figure(figsize=(10, 5))
sns.boxplot(data=df[numeric_cols])
plt.title('Boxplot of Numeric Variables')
plt.xticks(rotation=45)
plt.show()


In [ ]:
columns_to_plot = [col for col in df.columns if 'Volume' not in col]
plt.figure(figsize=(14, 8))
sns.boxplot(data=df[columns_to_plot])
plt.title('Boxplot of Numeric Variables (Excluding Volume)', fontsize=15)
plt.xticks(rotation=45)
plt.ylabel('Value')
plt.show()

In [ ]:
# Rolling MAD Outlier Detection
window = 30
threshold = 3.5
rolling_outliers = {}
for col in numeric_cols:
    series = df[col]
    rolling_median = series.rolling(window=window, center=True).median()
    mad = (series - rolling_median).abs().rolling(window=window, center=True).median()
    modified_z = 0.6745 * (series - rolling_median) / mad
    mask = modified_z.abs() > threshold
    count = mask.sum()
    percent = (count / len(df)) * 100
    rolling_outliers[col] = {
        "count": count,
        "percent": percent
    }

# Display results
for col, stats in rolling_outliers.items():
    print(f"- {col}: {stats['count']} rolling-MAD outliers ({stats['percent']:.2f}%)")


In [ ]:
plt.figure(figsize=(12,5))
plt.plot(ts.index, ts[target_col], label='Original')
plt.scatter(
    ts.index[outlier_mask],
    ts[target_col][outlier_mask],
    color='red',
    label='STL Outliers',
    s=20
)
plt.title(f"{target_col} with STL-based Outliers")
plt.legend()
plt.show()


**Distribution of Numeric Columns**

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(12,4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()


In [ ]:
for col in numeric_cols:

    # Skip columns with too many NaNs
    if df[col].dropna().shape[0] < 10:
        continue

    # Returns instead of raw values
    returns = df[col].pct_change().dropna()

    # Histogram of returns
    plt.figure(figsize=(12,4))
    sns.histplot(returns, bins=50, kde=True)
    plt.title(f"Return Distribution of {col}")
    plt.xlabel("Returns")
    plt.ylabel("Frequency")
    plt.show()


In [ ]:
for col in numeric_cols:

    # Skip columns with too many NaNs
    if df[col].dropna().shape[0] < 10:
        continue
    # Q-Q plot
    sm.qqplot(returns, line='s')
    plt.title(f"Q-Q Plot of {col} Returns")
    plt.show()

**Time Series Plots**

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    plt.figure(figsize=(12,4))
    plt.plot(df['Date'], df[col], linewidth=1.5)
    plt.title(f'{col} Over Time')
    plt.ylabel(col)
    plt.xlabel('Date')
    plt.grid(True, alpha=0.3)
    plt.show()


**Correlation Heatmap**

In [ ]:
numeric_data = df.select_dtypes(include=[np.number])
correlation_matrix = df.corr(numeric_only=True)
correlation_matrix = numeric_data.corr(method='pearson')
plt.figure(figsize=(15,8))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.show()

**Time Series Decomposition**

In [ ]:
# STL(Seasonal-Trend decomposition)
ts = df[['Date', 'Price_Gold']].dropna().set_index('Date')

stl = STL(ts['Price_Gold'], period=365, robust=True)
res = stl.fit()

plt.figure(figsize=(14,10))

plt.subplot(3,1,1)
plt.plot(res.trend, label='Trend')
plt.legend()

plt.subplot(3,1,2)
plt.plot(res.seasonal, label='Seasonality')
plt.legend()

plt.subplot(3,1,3)
plt.plot(res.resid, label='Residuals')
plt.legend()

plt.show()


NameError: name 'df' is not defined

** italicized textStationarity Test**

In [ ]:
# ADF (Augmented Dickey-Fuller Test)
result = adfuller(df['Price_Gold'])
print('ADF Statistic:', result[0])
print('p-value:', result[1])
if result[1] < 0.05:
    print("The series is stationary")
else:
    print("The series is NOT stationary")


In [ ]:
# KPSS (Kwiatkowski-Phillips-Schmidt-Shin)
from statsmodels.tsa.stattools import kpss

stat, p_value, _, _ = kpss(df['Price_Gold'], regression='c')
print('KPSS Statistic:', stat)
print('p-value:', p_value)

if p_value < 0.05:
    print("The series is NOT stationary")
else:
    print("The series is stationary")


**Autocorrelation & Partial Autocorrelation**

In [ ]:
plot_acf(df['Price_Gold'], lags=50)
plt.show()

plot_pacf(df['Price_Gold'], lags=50)
plt.show()


**Gold vs external indicators**

In [ ]:
features = ['Price_Oil', 'Price_Dollar', 'Price_Stocks']
for feature in features:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=df[feature], y=df['Price_Gold'])
    plt.title(f'Gold Price vs {feature}')
    plt.show()

**trend smoothing and Volatility**

In [ ]:
# Rolling statistics
roll_mean = df['Price_Gold'].rolling(window=30).mean()
roll_std = df['Price_Gold'].rolling(window=30).std()

# Plot with dual y-axes
fig, ax1 = plt.subplots(figsize=(14, 7))

# Left axis: Gold price & trend
ax1.plot(df['Date'], df['Price_Gold'], label='Gold Price')
ax1.plot(df['Date'], roll_mean, label='Rolling Mean (30 days)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Gold Price')
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)

# Right axis: Volatility
ax2 = ax1.twinx()
ax2.plot(df['Date'], roll_std, linestyle='--', label='Rolling Std (Volatility)')
ax2.set_ylabel('Volatility')
ax2.legend(loc='upper right')

plt.title('Gold Price Trend and Volatility (30-Day Rolling)')
plt.show()
